In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import PIL.Image as Image
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import save_image
from ignite.metrics import FID, InceptionScore

# set random seed for reproducibility
torch.manual_seed(0)

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")

print("Path to dataset files:", path)

In [ ]:
# Dataset and output paths
IMG_SIZE = 256
TRAIN_DIR = r"C:\Users\kelvin\.cache\kagglehub\datasets\paultimothymooney\chest-xray-pneumonia\versions\2\chest_xray\train"
TEST_DIR = r"C:\Users\kelvin\.cache\kagglehub\datasets\paultimothymooney\chest-xray-pneumonia\versions\2\chest_xray\test"

# Convert images to grayscale, resize, crop, and normalize to [0, 1]
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize(IMG_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
])

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.latent_dim = latent_dim

        # convolution spatial dim downscaler, shift dim to channel
        def down(input_ch, output_ch):

            return nn.Sequential(

                nn.Conv2d(input_ch, output_ch, 4, 2, 1),
                nn.BatchNorm2d(output_ch),
                nn.LeakyReLU()
            )

        # convolutional spatial dim upscaler, shift dim away from channel
        def up(input_ch, output_ch):

            return nn.Sequential(

                nn.ConvTranspose2d(input_ch, output_ch, 4, 2, 1),
                nn.BatchNorm2d(output_ch),
                nn.LeakyReLU()
            )

        # C: channels at bottleneck, S: spatial res at bottleneck
        self.C, self.S = 512, 4

        # flat_dim: # of features before projecting onto latent space
        flat_dim = self.C * self.S * self.S

        # initial compressor encoder (256x256->4x4)
        self.encoder = nn.Sequential(

            down(1, 32),
            down(32, 64),
            down(64, 128),
            down(128, 256),
            down(256, 512),
            down(512, self.C),
        )

        self.fc_mu = nn.Linear(flat_dim, latent_dim)
        self.fc_logvar = nn.Linear(flat_dim, latent_dim)
        self.fc_decoder = nn.Linear(latent_dim, flat_dim)

        # decoder reconstructing 256x256 image
        self.decoder = nn.Sequential(

            up(self.C, 512),
            up(512, 256),
            up(256, 128),
            up(128, 64),
            up(64, 32),
            nn.ConvTranspose2d(32, 1, 4, 2, 1),
            nn.Sigmoid(),
        )

    # (clamped [-10, 10] for stability)
    def encode(self, x):
        
        h = self.encoder(x).flatten(1)
        mu = self.fc_mu(h)
        
        logvar = torch.clamp(self.fc_logvar(h), -10, 10)
        return mu, logvar

    def reparameterize(self, mu, logvar):

        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + std * eps

    def decode(self, z):

        h = self.fc_decoder(z).view(-1, self.C, self.S, self.S)
        return self.decoder(h)

    # standard forwardpass
    def forward(self, x):
        
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

In [ ]:
# compute elbo loss
def vae_loss(x_hat, x, mu, logvar, beta=1.0):

    B = x.size(0)
    recon = F.binary_cross_entropy(x_hat, x, reduction="sum") / B
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / B
    return recon + beta * kl, recon, kl

# "GPU accelerated interpolation for Inception metrics" (honestly, not too sure how this works.)
# replaces an image resizing loop with a "GPU PyTorch tensor operation": directly resize
# images in VRAM with CUDA. Doesn't change results, just reduces training loop.
def interpolate(batch):
    # Expand 1-channel grayscale to 3 channels for Inception-v3
    if batch.shape[1] == 1:
        batch = batch.repeat(1, 3, 1, 1)
    # Perform bilinear interpolation directly on the GPU tensor
    return F.interpolate(batch, size=(299, 299), mode="bilinear", align_corners=False)

# compute metrics (fid, is)
def evaluate_fid_is(model, real_loader, device):
    
    fid_metric = FID(device=device)
    is_metric = InceptionScore(device=device)
    model.eval()

    with torch.no_grad():

        for i, (real, _) in enumerate(real_loader):
            real = real.to(device)

            # generate init fake x-ray from normal noise
            z = torch.randn(real.size(0), model.latent_dim, device=device)
            fake = model.decode(z)

            # accum. stats for final metric calculation
            fake_299 = interpolate(fake)
            real_299 = interpolate(real)
            
            fid_metric.update((fake_299, real_299))
            is_metric.update(fake_299)

            # log every 10 iters tracking for sanity
            if i % 10 == 0:
                print(f"eval batch {i}/{len(real_loader)}")

    return fid_metric.compute(), is_metric.compute()

In [ ]:
import gc
import os

# The parameter grid to test (9 combinations total)
latent_dims = [64, 128, 256]
betas = [0.5, 1.0, 2.0]
learning_rate = 1e-4
epochs = 50
batch_size = 32

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device} for Grid Search")

# Load Data ONCE to save I/O time across all 9 runs
train_set = datasets.ImageFolder(TRAIN_DIR, transform=transform)
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=0, drop_last=False)
test_set = datasets.ImageFolder(TEST_DIR, transform=transform)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=0)
real_loader = DataLoader(train_set, batch_size=64, shuffle=False, num_workers=0)

fixed_x, _ = next(iter(test_loader))
fixed_x = fixed_x[:8].to(device)
real_examples, _ = next(iter(DataLoader(train_set, batch_size=64, shuffle=True)))

# AUTOMATED GRID SEARCH LOOP
for dim in latent_dims:
    for beta in betas:

        # patch fix, crashed during this config (already recovered)
        if dim == 64 and beta == 0.5:
            continue

        print(f"\n{'='*50}")
        print(f"STARTING RUN: Latent Dim = {dim}, Beta = {beta}")
        print(f"{'='*50}")
        
        # 1. Setup output directory for this specific run
        OUT_DIR = f"./output/dim_{dim}_beta_{beta}_lr_{learning_rate}"
        os.makedirs(OUT_DIR, exist_ok=True)
        
        # 2. Initialize fresh model and optimizer for this run
        model = VAE(latent_dim=dim).to(device)
        optim = torch.optim.Adam(model.parameters(), lr=learning_rate)
        fixed_z = torch.randn(64, model.latent_dim, device=device)
        
        history = {"train_loss": [], "train_recon": [], "train_kl": [], "test_loss": []}

        # 3. Training Loop
        for i in range(epochs):
            model.train()
            sum_loss = sum_recon = sum_kl = 0.0
            for x, _ in train_loader:
                x = x.to(device)
                x_hat, mu, logvar = model(x)
                loss, recon, kl = vae_loss(x_hat, x, mu, logvar, beta)

                optim.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=100.0)
                optim.step()

                sum_loss += loss.item()
                sum_recon += recon.item()
                sum_kl += kl.item()

            n = len(train_loader)
            history["train_loss"].append(sum_loss / n)
            history["train_recon"].append(sum_recon / n)
            history["train_kl"].append(sum_kl / n)

            # Evaluation phase
            model.eval()
            test_loss = 0.0
            with torch.no_grad():
                for x, _ in test_loader:
                    x = x.to(device)
                    x_hat, mu, logvar = model(x)
                    loss, _, _ = vae_loss(x_hat, x, mu, logvar, beta)
                    test_loss += loss.item()
                history["test_loss"].append(test_loss / len(test_loader))

                # Save intermediate images every 10 epochs (keeps the folder clean)
                if (i+1) % 10 == 0 or i == epochs - 1:
                    samples = model.decode(fixed_z)
                    save_image(samples, f"{OUT_DIR}/samples_epoch{i+1:03d}.png", nrow=8)
                    recon_x, _, _ = model(fixed_x)
                    save_image(torch.cat([fixed_x, recon_x]), f"{OUT_DIR}/recon_epoch{i+1:03d}.png", nrow=8)

            print(f"Epoch {i+1:3d}/{epochs} | train {history['train_loss'][-1]:.1f} | test {history['test_loss'][-1]:.1f}")

        # 4. Save artifacts and plot curves
        torch.save(model.state_dict(), f"{OUT_DIR}/vae.pt")
        
        ep = range(1, epochs + 1)
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        axes[0].plot(ep, history["train_loss"], label="train")
        axes[0].plot(ep, history["test_loss"], label="test")
        axes[0].set_title(f"Total loss (dim:{dim}, beta:{beta})")
        axes[0].legend()
        axes[1].plot(ep, history["train_recon"])
        axes[1].set_title("Reconstruction (BCE)")
        axes[2].plot(ep, history["train_kl"])
        axes[2].set_title("KL divergence")
        for ax in axes: ax.set_xlabel("Epoch")
        plt.tight_layout()
        plt.savefig(f"{OUT_DIR}/loss_curves.png")
        plt.close()

        # 5. Compute Final Metrics
        print(f"Evaluating FID/IS for Dim={dim}, Beta={beta}...")
        model.eval()
        with torch.no_grad():
            samples = model.decode(torch.randn(64, model.latent_dim, device=device))
        save_image(samples, f"{OUT_DIR}/generated_final.png", nrow=8)
        save_image(real_examples, f"{OUT_DIR}/real_examples.png", nrow=8)

        fid, is_score = evaluate_fid_is(model, real_loader, device)
        with open(f"{OUT_DIR}/metrics.txt", "w") as f:
            f.write(f"FID: {fid:.4f}\nIS: {is_score:.4f}\n")
            
        # Free memory before the next loop begins
        del model, optim, history
        torch.cuda.empty_cache()
        gc.collect()

print("\nALL RUNS COMPLETED SUCCESSFULLY.")

In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image

dims = [64, 128]
betas = [0.5, 1.0, 2.0]
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

for i, dim in enumerate(dims):
    for j, beta in enumerate(betas):
        folder = f"./output/dim_{dim}_beta_{beta}_lr_0.0001"
        
        # 1. Print simple text metrics
        try:
            with open(f"{folder}/metrics.txt") as f:
                stats = f.read().strip().replace('\n', ', ')
                print(f"Dim {dim} | Beta {beta} -> {stats}")
        except FileNotFoundError:
            pass
            
        # 2. Add images to the visual grid
        try:
            axes[i, j].imshow(Image.open(f"{folder}/generated_final.png"))
        except FileNotFoundError:
            pass
        
        axes[i, j].set_title(f"Dim {dim} | Beta {beta}")
        axes[i, j].axis("off")

plt.tight_layout()
plt.savefig("./output/report_grid.png")
plt.show()